In [3]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langdetect import detect, LangDetectException
from typing import List, Tuple

# -------------------------------------------------------------------------
# Modelos
# -------------------------------------------------------------------------
# Inglês  → FinBERT-tone (3 classes: Positive / Negative / Neutral)
# Português → lxyuan/distilbert-base-multilingual-cased-sentiments-student
#             (3 classes: positive / negative / neutral — treinado em PT também)
# -------------------------------------------------------------------------

MODELOS = {
    "en": "yiyanghkust/finbert-tone",
    "pt": "lxyuan/distilbert-base-multilingual-cased-sentiments-student",
}

def carregar_modelo(nome: str):
    tok = AutoTokenizer.from_pretrained(nome)
    mod = AutoModelForSequenceClassification.from_pretrained(nome)
    mod.eval()
    return tok, mod

print("Carregando modelos...")
tok_en, mod_en = carregar_modelo(MODELOS["en"])
tok_pt, mod_pt = carregar_modelo(MODELOS["pt"])
print("Modelos prontos.")

# -------------------------------------------------------------------------
# Detecção de idioma
# -------------------------------------------------------------------------

def detectar_idioma(texto: str) -> str:
    """Retorna 'en', 'pt' ou 'other'. Fallback para 'en' se incerto."""
    try:
        lang = detect(texto)
        return lang if lang in ("en", "pt") else "en"  # fallback inglês
    except LangDetectException:
        return "en"

# -------------------------------------------------------------------------
# Inferência por lote, agnóstica ao modelo
# -------------------------------------------------------------------------

def inferir_lote(texts: List[str], tokenizer, model, batch_size: int = 32) -> np.ndarray:
    """Retorna array (N, num_classes) de probabilidades softmax."""
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

# -------------------------------------------------------------------------
# Score líquido: p(Positive) - p(Negative), normalizado para cada modelo
# -------------------------------------------------------------------------

def obter_indices(model) -> dict:
    """Mapeia label textual → índice, tolerante a variações de casing."""
    return {
        v.lower(): k
        for k, v in model.config.id2label.items()
    }

def score_liquido(probs_row: np.ndarray, indices: dict) -> Tuple[float, float]:
    """
    Retorna (score_liquido, confianca).
    score_liquido: p(pos) - p(neg), range [-1, +1]
    confianca:     probabilidade da classe dominante
    """
    p_pos = float(probs_row[indices["positive"]])
    p_neg = float(probs_row[indices["negative"]])
    p_neu = float(probs_row[indices["neutral"]])
    score = p_pos - p_neg
    conf  = max(p_pos, p_neg, p_neu)
    return score, conf

# -------------------------------------------------------------------------
# Limiares calibrados — mais generosos que ±0.10
#
# Por que mudar?  Títulos curtos (~8 palavras) raramente geram scores
# extremos. O FinBERT concentra massa em Neutral por design quando o
# texto não tem sinal forte. Limiares ±0.10 para neutro são muito
# estreitos — qualquer ruído de tokenização fica "neutro".
#
# Calibração sugerida após análise da distribuição real:
#   Veja o histograma de score_liquido e ajuste os breaks nos percentis
#   10/30/70/90 do seu dataset. Os valores abaixo são um bom ponto de
#   partida para títulos financeiros curtos.
# -------------------------------------------------------------------------

LIMIARES = [
    (-1.00, -0.35, "Muito negativo", -2),
    (-0.35, -0.10, "Negativo",        -1),
    (-0.10,  0.10, "Neutro",           0),
    ( 0.10,  0.35, "Positivo",        +1),
    ( 0.35,  1.00, "Muito positivo",  +2),
]

def grau_sentimento(score: float) -> Tuple[str, int]:
    for low, high, label, valor in LIMIARES:
        if low <= score < high:
            return label, valor
    return "Muito positivo", +2

# -------------------------------------------------------------------------
# Pipeline principal
# -------------------------------------------------------------------------

def classificar_textos(texts: List[str]) -> pd.DataFrame:
    # 1. Detecta idioma de cada título
    idiomas = [detectar_idioma(t) for t in texts]

    # 2. Separa índices por idioma para inferência em lote
    idx_en = [i for i, l in enumerate(idiomas) if l == "en"]
    idx_pt = [i for i, l in enumerate(idiomas) if l == "pt"]

    probs_all   = np.zeros((len(texts), 3))
    indices_all = [None] * len(texts)

    for idx_list, tok, mod in [
        (idx_en, tok_en, mod_en),
        (idx_pt, tok_pt, mod_pt),
    ]:
        if not idx_list:
            continue
        batch_texts = [texts[i] for i in idx_list]
        probs       = inferir_lote(batch_texts, tok, mod)
        indices     = obter_indices(mod)
        for j, i in enumerate(idx_list):
            probs_all[i]   = probs[j]
            indices_all[i] = indices

    # 3. Monta resultado
    resultados = []
    for i, texto in enumerate(texts):
        idx     = indices_all[i]
        p_pos   = float(probs_all[i, idx["positive"]])
        p_neg   = float(probs_all[i, idx["negative"]])
        p_neu   = float(probs_all[i, idx["neutral"]])
        score   = p_pos - p_neg
        conf    = max(p_pos, p_neg, p_neu)
        label, valor = grau_sentimento(score)

        resultados.append({
            "texto":           texto,
            "idioma":          idiomas[i],
            "grau_sentimento": label,
            "valor_numerico":  valor,
            "score_liquido":   round(score, 4),
            "confianca_pct":   round(conf * 100, 1),
            "prob_positivo":   round(p_pos, 4),
            "prob_negativo":   round(p_neg, 4),
            "prob_neutro":     round(p_neu, 4),
        })

    return pd.DataFrame(resultados)

# -------------------------------------------------------------------------
# Diagnóstico de distribuição — rode isso primeiro para calibrar limiares
# -------------------------------------------------------------------------

def diagnosticar_distribuicao(df_resultado: pd.DataFrame) -> None:
    """Imprime distribuição de scores e classes — use para calibrar limiares."""
    print("\n=== Distribuição de scores líquidos ===")
    print(df_resultado["score_liquido"].describe().round(3))

    print("\n=== Percentis ===")
    for p in [10, 25, 50, 75, 90]:
        v = df_resultado["score_liquido"].quantile(p / 100)
        print(f"  P{p:2d}: {v:.3f}")

    print("\n=== Contagem por grau ===")
    print(df_resultado["grau_sentimento"].value_counts())

    print("\n=== Contagem por idioma ===")
    print(df_resultado["idioma"].value_counts())

# -------------------------------------------------------------------------
# Uso
# -------------------------------------------------------------------------

df = pd.read_csv("noticias.csv")
textos = df["titulo"].tolist()

df_resultado = classificar_textos(textos)

# Diagnóstico antes de aceitar os limiares
diagnosticar_distribuicao(df_resultado)

# Só depois de ver os percentis — ajuste LIMIARES se necessário e reclassifique
df_final = df.copy()
df_final["idioma"]           = df_resultado["idioma"]
df_final["grau_sentimento"]  = df_resultado["grau_sentimento"]
df_final["valor_numerico"]   = df_resultado["valor_numerico"]
df_final["score_liquido"]    = df_resultado["score_liquido"]
df_final["confianca_pct"]    = df_resultado["confianca_pct"]

df_final.to_excel("noticias_classificadas.xlsx", index=False)
print("\nSalvo em noticias_classificadas.xlsx")

Carregando modelos...
Modelos prontos.

=== Distribuição de scores líquidos ===
count    1444.000
mean        0.010
std         0.405
min        -1.000
25%        -0.217
50%         0.000
75%         0.265
max         1.000
Name: score_liquido, dtype: float64

=== Percentis ===
  P10: -0.544
  P25: -0.217
  P50: 0.000
  P75: 0.265
  P90: 0.516

=== Contagem por grau ===
grau_sentimento
Neutro            411
Positivo          274
Muito positivo    272
Muito negativo    247
Negativo          240
Name: count, dtype: int64

=== Contagem por idioma ===
idioma
pt    1117
en     327
Name: count, dtype: int64

Salvo em noticias_classificadas.xlsx
